# 01. OpenAI API 기초
**SK하이닉스 Autonomous R&D — Day 3** 

> ChatGPT 웹과 **같은 모델**을 코드에서 호출.

---
## 0. 라이브러리 설치

아래 셀을 **최초 1회** 실행.

In [5]:
!pip install openai==1.58.1

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 14.5 MB/s  0:00:00

   ------------- --------------------------  5/15 [idna]
   -------------------------- ------------- 10/15 [pydantic]
   -------------------------- ------------- 10/15 [pydantic]
   ----------------------------- ---------- 11/15 [httpcore]
   ---------------------------------- ----- 13/15 [httpx]
   ------------------------------------- -- 14/15 [openai]
   ------------------------------------- -- 14/15 [openai]
   ------------------------------------- -- 14/15 [openai]
   ------------------------------------- -- 14/15 [openai]
   ---------------------------------------- 15/15 [openai]



In [6]:
!pip install python-dotenv

---
## 1. LLM API란?

강의자료 요약:

| 구분 | ChatGPT 웹 | API |
|------|------------|-----|
| 사용자 | 사람이 직접 입력 | **프로그램**이 요청 |
| 결과 | 화면에 표시 | **response 객체**로 반환 |
| 본질 | 대화 | **Prompt → Text** |

API 요청의 핵심 3요소:
1. **`model`** — 어떤 LLM을 쓸지
2. **`messages`** — system / user (대화 내용)
3. **`temperature`** — 답변의 무작위성 (0=일관, 1=창의)

---
## 2. API 키 연결


처음에는 **키를 변수에 넣어** 바로 연결해 봅니다.

> OpenAI Platform → [API keys](https://platform.openai.com/api-keys) 에서 발급

In [ ]:
from openai import OpenAI

# 아래에 본인 키를 붙여넣고 실행
api_key = ''

client = OpenAI(api_key=api_key)

In [10]:
# 연결 테스트
test = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': 'Hi, reply with one word: OK'}],
    max_tokens=10,
)
print('연결 성공:', test.choices[0].message.content)

연결 성공: OK


1. **`.env` 파일**에 키 저장 (코드와 분리) (OPENAI_API_KEY=sk...)
2. **`.gitignore`**에 `.env` 등록 → Git에 **절대 올리지 않음**

In [16]:
from pathlib import Path

root = Path('..')  # ttest/
env_path = root / '.env'
gitignore_path = root / '.gitignore'

print('.env 존재:', env_path.exists(), '→', env_path.resolve())
print('.gitignore 존재:', gitignore_path.exists())

if gitignore_path.exists():
    content = gitignore_path.read_text(encoding='utf-8')
    print('.env in .gitignore:', '.env' in content)

.env 존재: True → C:\Users\finey\OneDrive\Desktop\실습폴더\.env
.gitignore 존재: True
.env in .gitignore: True


In [12]:
from dotenv import load_dotenv
import os

load_dotenv(root / '.env')
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    raise ValueError('ttest/.env에 OPENAI_API_KEY=sk-... 를 설정하세요.')

client = OpenAI(api_key=api_key)
print('✅ .env에서 API 키 로드 완료 (코드에 키 없음)')

✅ .env에서 API 키 로드 완료 (코드에 키 없음)


---
## 3. 첫 API 호출 

`chat.completions.create()` — **messages** 리스트를 보내면 AI가 답변합니다.

| role | 역할 |
|------|------|
| `system` | AI 역할·규칙 (선택) |
| `user` | 사용자 질문 |

In [17]:
response = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.1,
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user',   'content': '2002년 월드컵 우승 팀이 어디야?'},
    ],
)

In [25]:
response.choices[0].message.content

'2002년 월드컵 우승 팀은 브라질입니다. 브라질은 결승에서 독일을 2-0으로 이기고 5번째 월드컵 트로피를 차지했습니다.'

In [30]:
####### 2022년 월드컵 우승팀 물어보기

response = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.1,
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user',   'content': '2026년 월드컵 한국 vs 멕시코 누가 이겼어?'},
    ],
)



In [33]:
response

ChatCompletion(id='chatcmpl-DuBTrX6SeahcDVUg8uMXQQTblbXJm', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='2026년 월드컵 한국과 멕시코의 경기는 아직 진행되지 않았습니다. 2026년 월드컵은 2026년에 개최될 예정이며, 그 결과는 그때가 되어야 알 수 있습니다. 현재로서는 어떤 팀이 이길지 예측할 수 없습니다.', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None, annotations=[]))], created=1782282871, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_8d97543466', usage=CompletionUsage(completion_tokens=65, prompt_tokens=35, total_tokens=100, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [31]:
print(response.choices[0].message.content)

2026년 월드컵 한국과 멕시코의 경기는 아직 진행되지 않았습니다. 2026년 월드컵은 2026년에 개최될 예정이며, 그 결과는 그때가 되어야 알 수 있습니다. 현재로서는 어떤 팀이 이길지 예측할 수 없습니다.


---
## 4. 응답(Response) 객체

API는 **텍스트만**이 아니라 **객체**를 돌려줍니다.

| 필드 | 의미 |
|------|------|
| `response.choices[0].message.content` | **실제 답변 텍스트** ← 가장 많이 사용 |
| `response.choices[0].finish_reason` | `stop`=정상 종료, `length`=토큰 초과 |
| `response.usage.total_tokens` | 이번 호출에 쓴 토큰 수 (비용 참고) |
| `response.id` | 요청 ID (로그·디버깅용) |

In [32]:
print('ID:', response.id)
print('finish_reason:', response.choices[0].finish_reason)
print('답변:', response.choices[0].message.content)
print('토큰 사용:', response.usage.total_tokens, '(prompt:', response.usage.prompt_tokens,
      '/ completion:', response.usage.completion_tokens, ')')

ID: chatcmpl-DuBTrX6SeahcDVUg8uMXQQTblbXJm
finish_reason: stop
답변: 2026년 월드컵 한국과 멕시코의 경기는 아직 진행되지 않았습니다. 2026년 월드컵은 2026년에 개최될 예정이며, 그 결과는 그때가 되어야 알 수 있습니다. 현재로서는 어떤 팀이 이길지 예측할 수 없습니다.
토큰 사용: 100 (prompt: 35 / completion: 65 )


---
## 5. System / User 프롬프트 

같은 `user` 질문이라도 **`system`에 역할·출력 규칙**을 주면 답변 **형식·깊이·관점**이 달라집니다.

아래는 **의도적으로 짧고 모호한 질문**을 사용합니다. system 유무에 따라 차이가 잘 보입니다.

| | system | user |
|---|--------|------|
| 역할 | 모델의 **역할·규칙·형식** | 이번 **질문·작업** |
| 비유 | 직무 기술서 | 오늘 업무 지시 |

In [34]:
# 질문을 짧고 모호하게 → system이 없으면 범용 답변, 있으면 역할·형식에 맞춘 답변
question = 'for문이 뭐야?'

r1 = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.2,
    messages=[{'role': 'user', 'content': question}],
)

print('=== system 없음 (범용 설명, 길어질 수 있음) ===')
print(r1.choices[0].message.content)

=== system 없음 (범용 설명, 길어질 수 있음) ===
`for`문은 프로그래밍에서 반복문 중 하나로, 특정 조건에 따라 코드 블록을 여러 번 실행할 수 있게 해주는 구조입니다. 주로 리스트, 배열, 또는 범위와 같은 반복 가능한 객체를 순회(iterate)할 때 사용됩니다.

예를 들어, Python에서 `for`문을 사용하는 기본적인 예시는 다음과 같습니다:

```python
# 0부터 4까지의 숫자를 출력
for i in range(5):
    print(i)
```

위의 코드에서 `range(5)`는 0부터 4까지의 숫자를 생성하고, `for`문은 이 숫자들을 하나씩 `i`에 할당하여 반복적으로 `print(i)`를 실행합니다.

`for`문은 다양한 프로그래밍 언어에서 비슷한 방식으로 사용되며, 각 언어마다 문법이 조금씩 다를 수 있습니다.


In [35]:
r2 = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.2,
    messages=[
        {
            'role': 'system',
            'content': '''You are a Python tutor.
Rules:
- Answer in Korean only
- Use exactly 3 lines: [비유 1문장] / [코드 예시 max 3 lines] / [실무 한 줄]
- No long explanation, no bullet lists''',
        },
        {'role': 'user', 'content': question},
    ],
)

print('=== system 있음 (튜터 + 3줄 형식 강제) ===')
print(r2.choices[0].message.content)

=== system 있음 (튜터 + 3줄 형식 강제) ===
for문은 반복하는 기계처럼 여러 번 작업을 수행하는 도구입니다.  
```python
for i in range(5):
    print(i)
```
반복적인 작업을 간편하게 처리할 수 있습니다.


---
## 6. temperature — **일반 예시**

같은 질문, 다른 `temperature` → 결과가 어떻게 달라지는지 확인합니다.

| temperature | 특징 | 용도 |
|-------------|------|------|
| 0 ~ 0.3 | 일관적, 사실 위주 | 분석, 판정, 코드 |
| 0.7 ~ 1.0 | 다양·창의적 | 브레인스토밍, 카피 |

In [36]:
question = '팀 워크숍 아이스브레이킹 아이디어 1가지만.'

for temp in [0.0, 0.7, 1.0]:
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=temp,
        messages=[{'role': 'user', 'content': question}],
    )
    print(f'[temperature={temp}] {r.choices[0].message.content}')
    print()

[temperature=0.0] "두 진실과 한 거짓" 게임을 제안합니다. 각 팀원이 자신에 대한 두 가지 진실과 하나의 거짓을 말합니다. 다른 팀원들은 어떤 것이 거짓인지 맞춰보는 방식입니다. 이 게임은 서로에 대한 이해를 높이고, 자연스럽게 대화를 유도하는 데 도움이 됩니다. 재미있고 가벼운 분위기를 만들어 팀원 간의 친밀감을 증진시킬 수 있습니다.

[temperature=0.7] "두 진실과 한 거짓" 게임을 제안합니다. 

팀원 각자가 자신에 대한 두 가지 진실과 한 가지 거짓을 차례로 말합니다. 다른 팀원들은 어떤 것이 거짓인지 추측해야 합니다. 이 활동은 서로에 대한 이해를 높이고, 편안한 분위기를 조성하는 데 도움이 됩니다. 팀원들의 개성과 흥미로운 이야기를 공유할 수 있는 기회가 되기도 합니다.

[temperature=1.0] "두真 사실과 한 거짓" 게임을 해보세요. 각 팀원은 자기소개를 하면서 두 가지 사실과 한 가지 거짓을 말합니다. 다른 팀원들은 그 중 어떤 것이 거짓인지 맞추려고 합니다. 이 게임은 서로에 대해 더 알아갈 수 있는 좋은 기회를 제공하며, 분위기를 부드럽게 만드는 데 도움이 됩니다.



---
## 7. `max_tokens` — 출력 길이 제한

답변이 너무 길어지는 것을 막거나, 비용을 줄일 때 사용합니다.

In [37]:
for max_tok in [20, 100]:
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=0.3,
        max_tokens=max_tok,
        messages=[{'role': 'user', 'content': '대한민국의 역사를 요약해줘.'}],
    )
    print(f'--- max_tokens={max_tok} (finish: {r.choices[0].finish_reason}) ---')
    print(r.choices[0].message.content)
    print()

--- max_tokens=20 (finish: length) ---
대한민국의 역사는 고대부터 현대에 이르기까지 다양한 사건과 변화를 포함

--- max_tokens=100 (finish: length) ---
대한민국의 역사는 고대부터 현대에 이르기까지 다양한 사건과 변화를 겪어왔습니다. 아래는 대한민국 역사에 대한 간략한 요약입니다.

1. **고대 및 삼국 시대**: 
   - 한반도의 고대 역사는 고조선(기원전 2333년 경)으로 시작됩니다. 이후 백제, 신라, 고구려의 삼국 시대가 이어졌습니다. 이 시기에는 각



---
## 8. `ask()` 함수 — 재사용 패턴

같은 API 호출을 함수로 감싸기.

In [38]:
def ask(user_message, system_message='You are a helpful assistant.', temperature=0.2, max_tokens=None):
    """1턴 질의 — 답변 텍스트만 반환"""
    kwargs = dict(
        model='gpt-4o-mini',
        temperature=temperature,
        messages=[
            {'role': 'system', 'content': system_message},
            {'role': 'user',   'content': user_message},
        ],
    )
    if max_tokens is not None:
        kwargs['max_tokens'] = max_tokens
    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content


print(ask('서울 3일 여행 코스 추천해줘. bullet 3개만.', max_tokens=150))

서울 3일 여행 코스 추천드립니다:

1. **1일차: 역사와 문화 탐방**
   - 경복궁 방문: 조선 왕조의 대표적인 궁궐 탐방
   - 북촌 한옥마을: 전통 한옥과 골목길 산책
   - 인사동: 전통 찻집과 기념품 가게 탐방

2. **2일차: 현대와 예술의 만남**
   - 홍대 거리: 젊은 예술과 문화의 중심지 탐방
   - 연남동 카페 거리: 독특한 카페와 맛집 탐방
   - 서울시립미술관: 현대 미술


In [40]:
print(ask('안녕 내이름은 신범교야.', max_tokens=150))

안녕하세요, 신범교님! 만나서 반갑습니다. 어떻게 도와드릴까요?


In [41]:
print(ask('내이름이 뭐라고?', max_tokens=150))

죄송하지만, 당신의 이름을 알 수 있는 정보가 없습니다. 당신의 이름을 알려주시면 그에 맞춰 대화할 수 있습니다!


---
## 9. 멀티턴 대화 

이전 대화를 `messages`에 **그대로 이어 붙이면** 후속 질문이 가능합니다.

```
system → user → assistant → user → ...
```

### 기존대화

In [39]:
messages = [
    {'role': 'system', 'content': 'You are a helpful travel assistant. Answer in Korean.'},
    {'role': 'user',   'content': '제주도 2박 3일 여행 계획 짜줘. 하루 요약 1줄씩.'},
]

r1 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
answer1 = r1.choices[0].message.content
print('=== 1턴 ===')
print(answer1)

messages = [
    {'role': 'system', 'content': 'You are a helpful travel assistant. Answer in Korean.'},
    {'role': 'user',   'content': '2일차만 더 자세히. 맛집 2곳 포함.'},
]

r2 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
print('\n=== 2턴 (맥락 유지) ===')
print(r2.choices[0].message.content)

=== 1턴 ===
제주도 2박 3일 여행 계획은 다음과 같습니다:

**1일차:** 제주공항 도착 후 한라산 국립공원에서 하이킹을 즐기고, 저녁에는 제주시의 맛집에서 신선한 해산물을 맛본다.

**2일차:** 성산 일출봉에서 일출을 감상하고, 우도에서 자전거 투어 후, 저녁에는 제주 전통시장 탐방과 함께 지역 음식을 즐긴다.

**3일차:** 협재 해수욕장에서 해수욕과 해변 산책을 즐기고, 마지막으로 제주도 기념품 쇼핑 후 공항으로 출발한다.

=== 2턴 (맥락 유지) ===
물론입니다! 2일차 일정을 자세히 설명드리겠습니다.

### 2일차: 서울 탐방

#### 아침
- **조식**: 호텔에서 간단한 조식을 즐기세요. 서울의 많은 호텔에서는 한식과 양식을 모두 제공하니 취향에 맞게 선택하시면 좋습니다.

#### 오전
- **경복궁 방문**: 서울의 대표적인 고궁인 경복궁을 방문해보세요. 조선 왕조의 역사와 아름다운 건축물을 감상할 수 있습니다. 궁궐 내부를 둘러보며 가이드 투어를 이용하면 더 많은 정보를 얻을 수 있습니다.

#### 점심
- **맛집 1: 광화문 국밥**: 경복궁 근처에 위치한 이곳은 전통적인 한식 국밥을 제공합니다. 따뜻한 국밥 한 그릇으로 배를 채우고, 한국의 맛을 느껴보세요.

#### 오후
- **북촌 한옥마을 탐방**: 전통 한옥이 잘 보존된 북촌 한옥마을을 걸어보세요. 아름다운 골목길과 전통 가옥들을 구경하며 사진도 찍기 좋은 장소입니다. 이곳에서 한국의 전통 문화를 느낄 수 있습니다.

#### 저녁
- **맛집 2: 인사동 전통 찻집**: 인사동에 위치한 전통 찻집에서 저녁을 즐겨보세요. 다양한 전통 차와 함께 한국의 디저트인 떡이나 한과를 맛볼 수 있습니다. 아늑한 분위기에서 휴식을 취하며 하루를 마무리하기 좋습니다.

#### 저녁 후
- **인사동 거리 탐방**: 저녁 식사 후 인사동 거리를 산책하며 다양한 전통 공예품 가게와 갤러리를 구경해보세요. 기념품을 사기에도 좋은 장소입니다.

이렇게 2일차 일정을 구성해

### 멀티턴

### Assistant 메시지는 앞서 있었던 prompt들(user/system)에 대한 모델의 응답을 구성하며, 종종 대화 상태를 유지하기 위해 히스토리의 일부로 저장됩니다.

#### 특징
모델이 생성한 응답을 포함한다.

이전 message의 context 처리를 위해 대화의 연속성 유지를 돕는다.

다음 API 호출에서는 assistant 메시지를 포함시켜야 문맥이 이어짐.

In [47]:
messages = [
    {'role': 'system', 'content': 'You are a helpful travel assistant. Answer in Korean.'},
    {'role': 'user',   'content': '제주도 2박 3일 여행 계획 짜줘. 하루 요약 1줄씩.'},
]

r1 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
answer1 = r1.choices[0].message.content
print('=== 1턴 ===')
print(answer1)


=== 1턴 ===
제주도 2박 3일 여행 계획입니다.

**1일차:** 제주공항 도착 후 한라산 국립공원에서 하이킹하고, 저녁에는 제주 전통 음식인 흑돼지 바비큐를 즐깁니다.

**2일차:** 성산 일출봉에서 일출을 감상한 후, 우도 섬으로 가서 자전거 투어와 해변에서 여유로운 시간을 보냅니다.

**3일차:** 제주 민속촌과 천지연 폭포를 방문한 후, 공항으로 돌아가기 전 마지막으로 제주 특산물 쇼핑을 합니다.


In [48]:
messages

[{'role': 'system',
  'content': 'You are a helpful travel assistant. Answer in Korean.'},
 {'role': 'user', 'content': '제주도 2박 3일 여행 계획 짜줘. 하루 요약 1줄씩.'}]

In [49]:
messages.append({'role': 'assistant', 'content': answer1})
messages.append({'role': 'user', 'content': '2일차만 더 자세히. 맛집 2곳 포함.'})

In [52]:
messages

[{'role': 'system',
  'content': 'You are a helpful travel assistant. Answer in Korean.'},
 {'role': 'user', 'content': '제주도 2박 3일 여행 계획 짜줘. 하루 요약 1줄씩.'},
 {'role': 'assistant',
  'content': '제주도 2박 3일 여행 계획입니다.\n\n**1일차:** 제주공항 도착 후 한라산 국립공원에서 하이킹하고, 저녁에는 제주 전통 음식인 흑돼지 바비큐를 즐깁니다.\n\n**2일차:** 성산 일출봉에서 일출을 감상한 후, 우도 섬으로 가서 자전거 투어와 해변에서 여유로운 시간을 보냅니다.\n\n**3일차:** 제주 민속촌과 천지연 폭포를 방문한 후, 공항으로 돌아가기 전 마지막으로 제주 특산물 쇼핑을 합니다.'},
 {'role': 'user', 'content': '2일차만 더 자세히. 맛집 2곳 포함.'}]

In [51]:



r2 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
print('\n=== 2턴 (맥락 유지) ===')
print(r2.choices[0].message.content)


=== 2턴 (맥락 유지) ===
2일차 상세 계획입니다.

**아침:** 성산 일출봉에서 일출을 감상한 후, 근처의 **성산일출봉 해물뚝배기**에서 신선한 해물뚝배기로 아침 식사를 합니다.

**오전:** 성산 일출봉을 탐방한 후, 우도로 이동합니다. 우도에서는 자전거를 대여하여 섬을 둘러보며 아름다운 경치를 즐깁니다.

**점심:** 우도에서 유명한 **땅콩 아이스크림**과 함께 **우도 땅콩국수**를 맛보며 점심을 해결합니다.

**오후:** 우도의 해변에서 여유롭게 시간을 보내고, 해양 스포츠나 스노클링을 즐길 수 있습니다. 

**저녁:** 제주 본섬으로 돌아와 **제주 흑돼지 구이**를 맛볼 수 있는 **돈사돈**에서 저녁을 즐깁니다.

**밤:** 저녁 후, 제주 바다의 야경을 감상하며 하루를 마무리합니다.


In [44]:
messages

[{'role': 'system',
  'content': 'You are a helpful travel assistant. Answer in Korean.'},
 {'role': 'user', 'content': '제주도 2박 3일 여행 계획 짜줘. 하루 요약 1줄씩.'}]

---
## ✏️ 실습 1

아래 주제로 **2턴** 대화를 만들어 보세요.

1턴: "Python for문 기본 문법 설명해줘"
2턴: "방금 설명한 걸로 1~5 합 구하는 코드 예시만 보여줘"

In [ ]:
# ── 여기에 작성 ──
messages = [
    {'role': 'system', 'content': 'You are a Python tutor. Answer in Korean.'},
    # {'role': 'user', 'content': '...'},
]
# r1 = client.chat.completions.create(...)
# messages.append(...)
# r2 = client.chat.completions.create(...)

---
## ✏️ 실습 1

커서 프롬프트 : 3일차 폴더에 스트림릿과 실습코드의 openai api를 이용해서 간단한 챗봇을 만드는 python 코드 만들어줘

In [53]:
!pip install streamlit

  Using cached numpy-2.0.2-cp39-cp39-win_amd64.whl.metadata (59 kB)
   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   ----------- ---------------------------- 2.9/10.1 MB 15.2 MB/s eta 0:00:01
   ----------------- ---------------------- 4.5/10.1 MB 10.8 MB/s eta 0:00:01
   ----------------------- ---------------- 6.0/10.1 MB 9.5 MB/s eta 0:00:01
   ------------------------------ --------- 7.6/10.1 MB 8.9 MB/s eta 0:00:01
   ------------------------------------ --- 9.2/10.1 MB 8.5 MB/s eta 0:00:01
   ---------------------------------------- 10.1/10.1 MB 8.2 MB/s  0:00:01
   ---------------------------------------- 0.0/731.2 kB ? eta -:--:--
   ---------------------------------------- 731.2/731.2 kB 15.1 MB/s  0:00:00
Using cached numpy-2.0.2-cp39-cp39-win_amd64.whl (15.9 MB)
   ---------------------------------------- 0.0/11.4 MB ? eta -:--:--
   ------ --------------------------------- 1.8/11.4 MB 8.4 MB/s eta 0:00:02
   ------------ --------------------------- 3